# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [10]:
import duckdb
print('duckdb version before:', duckdb.__version__)

duckdb version before: 2.0.0


In [11]:
!pip install -q -U duckdb

In [12]:
import os, getpass, duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
DATA_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

# One row per page: two 30-day windows, anchored on the latest date in the table.
df = con.sql(f"""
WITH b AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{DATA_PATH}'))
SELECT f.client_hash_id, f.content_hash_id,
  SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
  SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
  AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30,
  STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)    AS pos_std_prev30
FROM read_parquet('{DATA_PATH}') f, b
WHERE f.report_date > b.end_d - INTERVAL 60 DAY
GROUP BY 1, 2
HAVING imp_prev30 >= 100
""").df()

df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
print('pages:', len(df), '  decline rate:', round(df['is_declining'].mean(), 3))

# Distributions of key fields (aggregates only, no rows).
print(df[['imp_prev30', 'pos_prev30', 'pos_std_prev30']]
      .describe(percentiles=[.5, .9, .99]).round(2).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 111247   decline rate: 0.643
       imp_prev30  pos_prev30  pos_std_prev30
count   111247.00   111247.00       111240.00
mean      2290.66       19.88            8.54
std       6942.05       15.22            7.04
min        100.00        0.34            0.04
50%        565.00       14.68            6.82
90%       4973.00       41.21           18.57
99%      27631.54       71.45           29.11
max     585502.00       92.29          111.95


**Distributions (111,247 pages).** All three key fields have heavy right tails.
Previous-30-day impressions have a median of 565 but a mean of 2,291; the 99th percentile is
27,632 and the maximum is 585,502, so a small number of very large pages dominate any average.
Average position has a median of 14.7 (second page) and a 90th percentile of 41, so most pages
sit well below the top results. Position volatility (standard deviation of daily average
position in the earlier window) has a median of 6.8 and a 99th percentile of 29. Seven pages
have no volatility value. Because of these tails, I compare groups by quartile or band, not by
mean values.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [14]:
def by_group(frame, col, bins=None, q=None, labels=None):
    g = pd.qcut(frame[col], q=q, duplicates='drop') if q else pd.cut(frame[col], bins=bins, labels=labels)
    out = (frame.assign(group=g).groupby('group', observed=True)['is_declining']
           .agg(pages='size', decline_rate='mean').round(3))
    return out

print('Signal 1: prior volume (imp_prev30 quartiles)')
print(by_group(df, 'imp_prev30', q=4).to_string(), '\n')

print('Signal 2: prior position band')
print(by_group(df, 'pos_prev30', bins=[0, 3, 10, 20, 50, 100],
               labels=['1-3', '4-10', '11-20', '21-50', '51+']).to_string(), '\n')

print('Signal 3: position volatility (quartiles)')
print(by_group(df.dropna(subset=['pos_std_prev30']), 'pos_std_prev30', q=4).to_string(), '\n')

print('overall decline rate:', round(df['is_declining'].mean(), 3))

Signal 1: prior volume (imp_prev30 quartiles)
                    pages  decline_rate
group                                  
(99.999, 231.0]     27864         0.642
(231.0, 565.0]      27802         0.637
(565.0, 1801.0]     27776         0.649
(1801.0, 585502.0]  27805         0.645 

Signal 2: prior position band
       pages  decline_rate
group                     
1-3      631         0.331
4-10   36195         0.572
11-20  32917         0.654
21-50  35096         0.714
51+     6408         0.634 

Signal 3: position volatility (quartiles)
                   pages  decline_rate
group                                 
(0.0433, 2.934]    27810         0.579
(2.934, 6.823]     27810         0.659
(6.823, 12.383]    27810         0.677
(12.383, 111.947]  27810         0.658 

overall decline rate: 0.643


All three signals are measured in the earlier 30-day window, so they do not overlap the
window that defines decline (111,247 pages; overall decline rate 64.3%).

**Signal 1: prior volume, verdict FALSE.** I expected pages with more impressions to decline
less often. Decline rates across impression quartiles were 64.2%, 63.7%, 64.9% and 64.5%, so
volume carries no visible signal.

**Signal 2: prior position, verdict CONFIRMED (with a caveat).** I expected pages ranking
higher to decline less often. Decline rates rose from 33.1% at positions 1 to 3 (631 pages) to
57.2% at 4 to 10, 65.4% at 11 to 20 and 71.4% at 21 to 50. At position 51 and beyond the rate
fell back to 63.4%, so the relationship holds up to about position 50 and is not smooth
beyond it.

**Signal 3: position volatility, verdict MIXED.** I expected pages whose position swings
more to decline more often. The lowest-volatility quartile declined least (57.9%), but the
other three quartiles were similar (65.9%, 67.7%, 65.8%), so only stability appears to
protect and more volatility does not keep raising the rate. Volatility may partly reflect
position, since deeper pages move more. I did not test that.

These are observed, directional associations, not evidence of cause.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [16]:
extra = con.sql(f"""
WITH b AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{DATA_PATH}'))
SELECT f.client_hash_id, f.content_hash_id,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30
FROM read_parquet('{DATA_PATH}') f, b
WHERE f.report_date > b.end_d - INTERVAL 60 DAY
GROUP BY 1, 2
""").df()
d = df.merge(extra, on=['client_hash_id', 'content_hash_id'], how='left')

# Page-1 pages in the earlier window, grouped by position band and by volume.
p1 = d[d['pos_prev30'] <= 10].copy()
p1['ctr_pct'] = 100 * p1['clk_prev30'] / p1['imp_prev30']
p1['band'] = pd.cut(p1['pos_prev30'], [0, 3, 5, 7, 10], labels=['1-3', '3-5', '5-7', '7-10'])
p1['volume'] = pd.cut(p1['imp_prev30'], [99, 200, 1000, 5000, float('inf')],
                      labels=['100-200', '200-1k', '1k-5k', '5k+'])

print('Median CTR (%) by position band and impression volume:')
print(p1.pivot_table(index='band', columns='volume', values='ctr_pct',
                     aggfunc='median', observed=True).round(2).to_string())

print('\nPages in each cell:')
print(p1.pivot_table(index='band', columns='volume', values='ctr_pct',
                     aggfunc='size', observed=True).to_string())

print('\nShare of pages with zero clicks:')
print(p1.assign(zero=(p1['clk_prev30'] == 0)).pivot_table(
    index='band', columns='volume', values='zero', aggfunc='mean', observed=True).round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Median CTR (%) by position band and impression volume:
volume  100-200  200-1k  1k-5k   5k+
band                                
1-3        0.00    0.49   0.80  0.70
3-5        0.57    0.56   0.59  0.48
5-7        0.00    0.45   0.40  0.29
7-10       0.00    0.32   0.27  0.17

Pages in each cell:
volume  100-200  200-1k  1k-5k   5k+
band                                
1-3          64     103    216   248
3-5         434     957   1675  1562
5-7        1243    2997   4451  2901
7-10       2551    7080   7095  3249

Share of pages with zero clicks:
volume  100-200  200-1k  1k-5k    5k+
band                                 
1-3       0.516   0.184  0.005  0.000
3-5       0.463   0.187  0.013  0.002
5-7       0.521   0.195  0.020  0.003
7-10      0.622   0.283  0.057  0.009


**Signal tested:** CTR benchmarked by position, the logic behind CTR-style flags. The data
dictionary warns that tier CTRs need a volume floor, so I tested the assumption directly: is
one CTR benchmark per position band comparable across pages of different size? (Page-1 pages,
earlier 30-day window, median per-page CTR. These are per-page medians, not pooled rates, so
they are not comparable with warehouse-wide CTR figures.)

**Result.** For pages with 100 to 200 impressions, median CTR was 0.00% in three of the four
position bands, and 46% to 62% of those pages had zero clicks, so a benchmark flags many of
them by chance. From 1,000 impressions upward, zero-click pages are rare (under 6%). Even among
larger pages, volume shifts the benchmark: at positions 7 to 10 the median CTR was 0.32%, 0.27%
and 0.17% for 200 to 1k, 1k to 5k and 5k+ impressions, and at positions 5 to 7 it fell from 0.45%
to 0.29%. Positions 3 to 5 were steadier (0.48% to 0.59%).

**Verdict: MIXED.** The assumption fails for small pages and holds only roughly for larger
ones. These are observed, directional medians, not an explanation of why volume shifts CTR.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

Before flagging a page for low click-through, set a volume floor of about 1,000 impressions in the window and compare it with pages of similar position and similar volume, not one benchmark per position band. Treat zero-click pages with a few hundred impressions as noise, since about half of them have no clicks by chance. Position alone is a useful signal in this data (pages deeper in the results declined more often), but prior volume told us nothing about decline.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.